# Generate feature-discovery dataset (English, Spanish, German)

In [2]:
import os
import sys
import time
from pathlib import Path

repo_root = Path.cwd().resolve()
if (repo_root / "src").is_dir():
    pass
elif (repo_root.parent / "src").is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import pandas as pd
from dotenv import load_dotenv

load_dotenv(repo_root / ".env")

from src.data.discovery_generation import (
    DiscoveryLangKey,
    DiscoveryTemplate,
    assign_source_indices_from_pair_ids,
    call_model_for_pairs,
    trim_to_valid_pairs,
    get_discovery_config,
    records_to_long_tsv,
)
from src.data.discovery_pairs import load_discovery_pairs_from_path

assert os.environ.get("OPENAI_API_KEY"), (
    "Set OPENAI_API_KEY before running (e.g. in .env)."
)

pd.set_option("display.max_colwidth", 100)

In [ ]:
DISCOVERY_LANGUAGE: DiscoveryLangKey = "en"  # "en" | "spa" | "deu"

MODEL = "gpt-5-mini"
REASONING_EFFORT = "medium"
PAIRS_PER_TEMPLATE = 10  # valid pairs to keep per template
OVERSAMPLE_PAIRS = 5  # ask the model for this many extra; first PAIRS_PER_TEMPLATE valid rows are kept
REQUEST_PAIRS_PER_TEMPLATE = PAIRS_PER_TEMPLATE + OVERSAMPLE_PAIRS
PAIR_ID_START = 0
DELAY = 1  # seconds between API calls

_cfg = get_discovery_config(DISCOVERY_LANGUAGE)
TEMPLATES_TO_USE: tuple[DiscoveryTemplate, ...] = _cfg.templates

EXTRA_HINT = """
Use varied, non-repetitive vocabulary across pairs. Do not reuse the same noun+verb lemma within this batch. It is extremely important to make the generated items as structurally and semantically as possible.
"""

OUT = repo_root / "data" / "feature_discovery_dataset" / {
    "en": "eng_generated.tsv",
    "spa": "spa_generated.tsv",
    "deu": "deu_generated.tsv",
}[DISCOVERY_LANGUAGE]

## 1. Call the model (one batch per template)

Each response is JSON `{"pairs": [ ... ]}`.

The notebook requests `PAIRS_PER_TEMPLATE` + `OVERSAMPLE_PAIRS`, then keeps the first `PAIRS_PER_TEMPLATE` rows that pass validation (filtering out bad JSON fields, identical prefixes/continuations, etc.).

In [ ]:
all_records: list = []
next_pair_num = PAIR_ID_START

for i, tmpl in enumerate(TEMPLATES_TO_USE):
    raw = call_model_for_pairs(
        n_pairs=REQUEST_PAIRS_PER_TEMPLATE,
        pair_id_start=next_pair_num,
        model=MODEL,
        reasoning_effort=REASONING_EFFORT,
        template=tmpl,
        diversity_hint=EXTRA_HINT,
        discovery_language=DISCOVERY_LANGUAGE,
        max_output_tokens=50000
    )
    recs, _dropped = trim_to_valid_pairs(
        raw,
        target_n=PAIRS_PER_TEMPLATE,
        pair_id_prefix=_cfg.pair_id_prefix,
        pair_id_start=next_pair_num,
    )
    print(
        f"[{i + 1}/{len(TEMPLATES_TO_USE)}] {tmpl.id}: "
        f"kept {len(recs)}/{PAIRS_PER_TEMPLATE} valid (API returned {len(raw)} rows)"
    )
    all_records.extend(recs)
    next_pair_num += PAIRS_PER_TEMPLATE
    if i + 1 < len(TEMPLATES_TO_USE):
        time.sleep(DELAY)

len(all_records)

## 2. Expand to TSV rows and save

In [6]:
assign_source_indices_from_pair_ids(all_records)

df = records_to_long_tsv(all_records, language=_cfg.tsv_language)
df = df.sort_values(["pair_id", "target_number"]).reset_index(drop=True)
df.to_csv(OUT, sep="\t", index=False)
print("Wrote", OUT, "—", len(df), "rows", "(", df["pair_id"].nunique(), "pairs)")
df.head(6)

Wrote /Users/veronicasmilga/thesis/data/feature_discovery_dataset/eng_generated.tsv — 270 rows ( 135 pairs)


,pair_id,language,target_number,good_prefix,bad_prefix,continuation,source_idx,distance,has_attractor,template_id,phenomenon_family
0,eng_0000,eng,PL,The doctors,The doctor,treat patients,0,1,0,simple_np,SV-#
1,eng_0000,eng,SG,The doctor,The doctors,treats patients,0,1,0,simple_np,SV-#
2,eng_0001,eng,PL,These committees,This committee,review the proposal,1,1,0,simple_np,SV-#
3,eng_0001,eng,SG,This committee,These committees,reviews the proposal,1,1,0,simple_np,SV-#
4,eng_0002,eng,PL,Tigers,A tiger,hunt deer,2,1,0,simple_np,SV-#
5,eng_0002,eng,SG,A tiger,Tigers,hunts deer,2,1,0,simple_np,SV-#


## 3. Validate (same merge as feature discovery)

In [7]:
def validate_discovery_tsv(path: Path) -> pd.DataFrame:
    pairs = load_discovery_pairs_from_path(path)
    raw = pd.read_csv(path, sep="\t")
    n_ids = raw["pair_id"].nunique()
    assert len(pairs) == n_ids, (len(pairs), n_ids)
    return pairs


validate_discovery_tsv(OUT)

2026-04-05 17:23:02,583 | src.data.discovery_pairs | INFO | Loaded 12 standardized discovery pairs from /Users/veronicasmilga/thesis/data/feature_discovery_dataset/eng_generated.tsv


,pair_id,language,prefix_sg,prefix_pl,continuation_sg,continuation_pl,source_idx_sg,source_idx_pl,distance,has_attractor
0,eng_0000,eng,The gardener,The gardeners,waters the lawn.,water the lawn.,0,0,1,0
1,eng_0001,eng,The anxious editor,The anxious editors,is reviewing the manuscript.,are reviewing the manuscript.,1,1,2,0
2,eng_0002,eng,The director of the city museum with extensive archival experience,The directors of the city museum with extensive archival experience,insists on reviewing the new donation immediately.,insist on reviewing the new donation immediately.,2,2,4,0
3,eng_0003,eng,The director of the reports,The directors of the reports,says that the team will review the proposal.,say that the team will review the proposal.,3,3,4,1
4,eng_0004,eng,The curator with the interns,The curators with the interns,is preparing a guided tour for visitors,are preparing a guided tour for visitors,4,4,5,1
5,eng_0005,eng,The experienced gardener in the courtyard,The experienced gardeners in the courtyard,has trimmed the rose bushes every spring.,have trimmed the rose bushes every spring.,5,5,3,0
6,eng_0006,eng,The child in the garden,The children in the garden,was playing in the sandbox.,were playing in the sandbox.,6,6,2,0
7,eng_0007,eng,The editor who interviewed the senators,The editors who interviewed the senator,is praising the committee's choice.,are praising the committee's choice.,7,7,5,1
8,eng_0008,eng,Peanut butter and jelly,Peanut butter and jellies,is commonly served at picnics,are commonly served at picnics,8,8,3,0
9,eng_0009,eng,The antique clock on the mantel,The antique clocks on the mantel,is in the living room.,are in the living room.,9,9,2,0


## 4. All languages

Run after the config cell. Generates and saves all three TSVs in one go. Pair numbering restarts at `PAIR_ID_START` for each language.


In [4]:
# Uses MODEL, PAIRS_PER_TEMPLATE, PAIR_ID_START, DELAY, EXTRA_HINT from the config cell.
OUT_PATHS: dict[DiscoveryLangKey, Path] = {
    "en": repo_root / "data" / "feature_discovery_dataset" / "eng_generated.tsv",
    "spa": repo_root / "data" / "feature_discovery_dataset" / "spa_generated.tsv",
    "deu": repo_root / "data" / "feature_discovery_dataset" / "deu_generated.tsv",
}

for lang in ("en", "spa", "deu"):
    cfg = get_discovery_config(lang)
    templates = cfg.templates
    all_records: list = []
    next_num = PAIR_ID_START
    print(f"=== {lang} ({len(templates)} templates) ===")
    for i, tmpl in enumerate(templates):
        raw = call_model_for_pairs(
            n_pairs=REQUEST_PAIRS_PER_TEMPLATE,
            pair_id_start=next_num,
            model=MODEL,
            reasoning_effort=REASONING_EFFORT,
            template=tmpl,
            diversity_hint=EXTRA_HINT,
            discovery_language=lang,
            max_output_tokens=50000
        )
        if not raw:
            print(
                f"  [{i + 1}/{len(templates)}] {tmpl.id}: 0 rows "
                f"(empty API body — often max_output_tokens / incomplete response; retry this template)"
            )
            recs = []
        else:
            recs, _dropped = trim_to_valid_pairs(
                raw,
                target_n=PAIRS_PER_TEMPLATE,
                pair_id_prefix=cfg.pair_id_prefix,
                pair_id_start=next_num,
            )
            print(
                f"  [{i + 1}/{len(templates)}] {tmpl.id}: "
                f"kept {len(recs)}/{PAIRS_PER_TEMPLATE} valid ({len(raw)} API rows)"
            )
        all_records.extend(recs)
        next_num += PAIRS_PER_TEMPLATE
        if i + 1 < len(templates):
            time.sleep(DELAY)
    assign_source_indices_from_pair_ids(all_records)
    df = records_to_long_tsv(all_records, language=cfg.tsv_language)
    df = df.sort_values(["pair_id", "target_number"]).reset_index(drop=True)
    out = OUT_PATHS[lang]
    df.to_csv(out, sep="\t", index=False)
    print(f"Wrote {out} — {len(df)} rows ({df['pair_id'].nunique()} pairs)\n")


=== en (10 templates) ===
  [1/10] simple_np: kept 10/10 valid (15 API rows)
  [2/10] adj_np: kept 10/10 valid (15 API rows)
  [3/10] long_subject_no_attractor: kept 10/10 valid (15 API rows)
  [4/10] pp_attractor: kept 10/10 valid (15 API rows)
  [5/10] quantifier_head_attractor: kept 10/10 valid (15 API rows)
  [6/10] perfect_aux: kept 10/10 valid (15 API rows)
  [7/10] irregular_plural: kept 10/10 valid (15 API rows)
  [8/10] relative_clause_subject: kept 10/10 valid (15 API rows)
  [9/10] non_initial_subject: kept 10/10 valid (15 API rows)
  [10/10] pronoun_subject: kept 10/10 valid (15 API rows)
Wrote /Users/veronicasmilga/thesis/data/feature_discovery_dataset/eng_generated.tsv — 200 rows (100 pairs)

=== spa (10 templates) ===
  [1/10] simple_np: kept 10/10 valid (15 API rows)
  [2/10] adj_np: kept 10/10 valid (15 API rows)
  [3/10] long_subject_no_attractor: kept 10/10 valid (15 API rows)
  [4/10] pp_attractor: kept 10/10 valid (15 API rows)
  [5/10] quantifier_head_attractor: k